
# BERTopic – Environment Dimension (Independent & Optimized)

Generated on: 2026-02-21 14:24:46

This notebook builds a fully independent BERTopic model for the Environment dimension of Sustainable Fashion.

Focus:
- Carbon emissions & footprint
- Waste & landfill reduction
- Circular fashion & recycling
- Renewable energy
- Eco-friendly materials

No forced number of topics (HDBSCAN determines clusters).


In [ ]:

!pip -q install bertopic sentence-transformers umap-learn hdbscan openpyxl emoji tqdm langdetect


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 22.0 MB/s eta 0:00:00


In [ ]:

import pandas as pd
import re
import emoji
from tqdm.auto import tqdm
from langdetect import detect

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [ ]:

# ============================
# LOAD DATA
# ============================
DATA_PATH = "/content/data clean environment stage 1.xlsx"

df = pd.read_excel(DATA_PATH)

candidate_cols = ["clean_text", "text", "tweet", "content"]
text_col = next((c for c in candidate_cols if c in df.columns), df.columns[0])

docs_raw = df[text_col].dropna().astype(str).tolist()

print("Text column:", text_col)
print("Raw documents:", len(docs_raw))


Text column: clean_text
Raw documents: 6325


In [ ]:

# ============================
# CLEANING
# ============================
URL_RE = re.compile(r"http\S+|www\S+")
MULTI_SPACE_RE = re.compile(r"\s+")

def clean_text(s: str) -> str:
    s = s.lower().strip()
    s = URL_RE.sub(" ", s)
    s = emoji.replace_emoji(s, replace=" ")
    s = re.sub(r"[^a-z\s]", " ", s)
    s = MULTI_SPACE_RE.sub(" ", s).strip()
    return s

docs = [clean_text(d) for d in docs_raw]
docs = [d for d in docs if len(d.split()) >= 5]

print("After cleaning:", len(docs))


After cleaning: 5544


In [ ]:

# ============================
# ENGLISH FILTER
# ============================
def filter_english(docs):
    filtered = []
    for d in tqdm(docs):
        try:
            if detect(d) == "en":
                filtered.append(d)
        except:
            continue
    return filtered

docs = filter_english(docs)
print("After EN filter:", len(docs))


  0%|          | 0/5544 [00:00<?, ?it/s]

After EN filter: 5410


In [ ]:

# ============================
# ENVIRONMENT STOPWORDS
# ============================
generic_fillers = {
    "think","need","make","people","just","like","really","also",
    "im","ive","dont","didnt","doesnt","cant","wont","get","got"
}

remove_economic = {"profit","price","prices","cost","costs","market","wage","salary","economy","economic"}
remove_social   = {"labour","labor","workers","sweatshop","slavery","rights","child","children"}
remove_cultural = {"heritage","tradition","cultural","indigenous"}
remove_aesthetic = {"style","design","designer","timeless","classic","quality","durable","minimal"}

fashion_general = {"brand","brands"}

stopwords = set(ENGLISH_STOP_WORDS) | generic_fillers | fashion_general | remove_economic | remove_social | remove_cultural | remove_aesthetic

vectorizer_model = CountVectorizer(
    stop_words=list(stopwords),
    ngram_range=(1,3),
    min_df=6,
    max_df=0.85
)

print("Stopwords size:", len(stopwords))


Stopwords size: 364


In [ ]:

# ============================
# MODEL COMPONENTS
# ============================
embedding_model = SentenceTransformer("all-mpnet-base-v2")

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=25,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:

# ============================
# SEED TOPICS (Environment)
# ============================
seed_topic_list = [
    ["carbon", "carbon footprint", "emissions", "co2", "climate change"],
    ["waste", "textile waste", "landfill", "pollution"],
    ["recycle", "recycling", "circular fashion", "circular economy"],
    ["renewable energy", "solar", "wind energy", "green energy"],
    ["organic cotton", "eco friendly", "natural fibers"]
]


In [ ]:

# ============================
# TRAIN BERTopic
# ============================
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    seed_topic_list=seed_topic_list,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

topic_info = topic_model.get_topic_info()
topic_info


2026-02-26 09:43:29,108 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/170 [00:00<?, ?it/s]

2026-02-26 09:52:42,810 - BERTopic - Embedding - Completed ✓
2026-02-26 09:52:42,812 - BERTopic - Guided - Find embeddings highly related to seeded topics.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-02-26 09:52:43,127 - BERTopic - Guided - Completed ✓
2026-02-26 09:52:43,128 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-26 09:53:29,440 - BERTopic - Dimensionality - Completed ✓
2026-02-26 09:53:29,441 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-26 09:53:30,270 - BERTopic - Cluster - Completed ✓
2026-02-26 09:53:30,287 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-26 09:53:30,643 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1766,-1_waste_buying_wear_things,"[waste, buying, wear, things, recycle, want, o...",[purchase new previously owned accept previous...
1,0,767,0_sustainable fashion_ethicalfashion_amp_ecofr...,"[sustainable fashion, ethicalfashion, amp, eco...",[embrace future fashion sustainable clothing s...
2,1,363,1_chemicals_toxic_landfill_waste,"[chemicals, toxic, landfill, waste, dyes, tras...",[yeseniah americans western imperial first wor...
3,2,258,2_china_factories_america_usa,"[china, factories, america, usa, italy, factor...",[reminds world economics forum discussions bou...
4,3,232,3_cotton_leather_wool_natural fibers,"[cotton, leather, wool, natural fibers, fiber,...",[also like fabrics skin wear natural fibers ot...
5,4,197,4_second hand_stores_second_thrift stores,"[second hand, stores, second, thrift stores, t...",[shopping second hand many years work clothes ...
6,5,193,5_emissions_carbon_carbon emissions_pollution,"[emissions, carbon, carbon emissions, pollutio...",[fast fashion damaging planet global carbon em...
7,6,185,6_jeans_years_shirt_bought,"[jeans, years, shirt, bought, wearing, years o...",[bought jeans shirts second hand store also un...
8,7,166,7_donate_goodwill_charity_donating,"[donate, goodwill, charity, donating, sell, pl...",[stopped donating thrift stores goodwill sell ...
9,8,145,8_recycling_recycle_landfill_uk,"[recycling, recycle, landfill, uk, clothing ye...",[know average american throws away lbs clothin...


In [ ]:

# ============================
# VISUALIZATION
# ============================
valid_topics = topic_info[topic_info["Topic"] != -1]["Topic"].tolist()

fig = topic_model.visualize_barchart(
    topics=valid_topics,
    n_words=10,
    title="Environment Topic Words Score"
)

fig.write_html("topic_barchart_Environment.html")
fig.show()


In [ ]:
# Visualisasi peta topik 2D yang interaktif
topic_model.visualize_topics()

In [ ]:
import pandas as pd
import numpy as np

# ============================
# BUILD SUMMARY TABLE
# ============================

# Total dokumen
total_docs = len(topics)

# Hitung jumlah dokumen per topic
topic_counts = pd.Series(topics).value_counts().sort_index()

# Hitung total tanpa noise (-1)
total_wo_noise = total_docs - topic_counts.get(-1, 0)

rows = []

for topic_id, n_doc in topic_counts.items():

    # Persentase dari total dokumen
    percent = (n_doc / total_docs) * 100

    # Ambil keywords top 5
    if topic_id != -1:
        words = topic_model.get_topic(topic_id)
        keywords = ", ".join([w[0] for w in words[:5]])

        # Persentase tanpa noise
        percent_wo_noise = (n_doc / total_wo_noise) * 100 if total_wo_noise > 0 else 0
    else:
        keywords = "Noise / Outliers"
        percent_wo_noise = None

    rows.append({
        "Topic": topic_id,
        "n_doc": n_doc,
        "%": round(percent, 2),
        "keywords": keywords,
        "%_wo_noise": round(percent_wo_noise, 2) if percent_wo_noise is not None else None
    })

summary_table = pd.DataFrame(rows).sort_values("Topic").reset_index(drop=True)

summary_table

,Topic,n_doc,%,keywords,%_wo_noise
0,-1,1766,32.64,Noise / Outliers,NaN
1,0,767,14.18,"sustainable fashion, ethicalfashion, amp, ecof...",21.05
2,1,363,6.71,"chemicals, toxic, landfill, waste, dyes",9.96
3,2,258,4.77,"china, factories, america, usa, italy",7.08
4,3,232,4.29,"cotton, leather, wool, natural fibers, fiber",6.37
5,4,197,3.64,"second hand, stores, second, thrift stores, th...",5.41
6,5,193,3.57,"emissions, carbon, carbon emissions, pollution...",5.30
7,6,185,3.42,"jeans, years, shirt, bought, wearing",5.08
8,7,166,3.07,"donate, goodwill, charity, donating, sell",4.56
9,8,145,2.68,"recycling, recycle, landfill, uk, clothing year",3.98


In [ ]:
summary_table["%"] = summary_table["%"].astype(str) + "%"
summary_table["%_wo_noise"] = summary_table["%_wo_noise"].astype(str) + "%"

summary_table

,Topic,n_doc,%,keywords,%_wo_noise
0,-1,1766,32.64%,Noise / Outliers,nan%
1,0,767,14.18%,"sustainable fashion, ethicalfashion, amp, ecof...",21.05%
2,1,363,6.71%,"chemicals, toxic, landfill, waste, dyes",9.96%
3,2,258,4.77%,"china, factories, america, usa, italy",7.08%
4,3,232,4.29%,"cotton, leather, wool, natural fibers, fiber",6.37%
5,4,197,3.64%,"second hand, stores, second, thrift stores, th...",5.41%
6,5,193,3.57%,"emissions, carbon, carbon emissions, pollution...",5.3%
7,6,185,3.42%,"jeans, years, shirt, bought, wearing",5.08%
8,7,166,3.07%,"donate, goodwill, charity, donating, sell",4.56%
9,8,145,2.68%,"recycling, recycle, landfill, uk, clothing year",3.98%


In [ ]:
summary_table.to_excel("bertopic_summary_table.xlsx", index=False)
print("✓ Saved: bertopic_summary_table.xlsx")

✓ Saved: bertopic_summary_table.xlsx


In [ ]:
from collections import Counter
from bertopic import BERTopic
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# ====== pilih topic environment core ======
selected_topics = [1, 5, 9, 14]
topic_mapping  = {1:0, 5:1, 9:2, 14:3}

# cek jumlah dokumen per topik biar gak zonk
cnt = Counter(topics)
for t in selected_topics:
    print(f"Old Topic {t} => {cnt.get(t,0)} docs")

# ====== filter dokumen ======
filtered_docs = []
filtered_old  = []
for doc, t in zip(docs, topics):
    if t in selected_topics:
        filtered_docs.append(doc)
        filtered_old.append(t)

print("Docs kept:", len(filtered_docs))
print("Old topics kept:", sorted(set(filtered_old)))

# ====== remap label jadi 0-3 ======
filtered_new = [topic_mapping[t] for t in filtered_old]

# Create a new vectorizer model for the compact model with adjusted parameters
# The original vectorizer_model's min_df=6 is too high for only 4 aggregated topic documents
core_vectorizer_model = CountVectorizer(
    stop_words=list(stopwords), # Use the same stopwords as before
    ngram_range=(1,3),
    min_df=1,  # Lowered for smaller number of aggregated topic documents (which is 4 here)
    max_df=1.0 # Adjusted for smaller number of aggregated topic documents, meaning allow words in all topics if needed
)

# ====== build compact model (label baru 0-3) ======
env_core_model = BERTopic(
    embedding_model=embedding_model, # Use the original embedding_model variable
    vectorizer_model=core_vectorizer_model, # Use the new vectorizer model
    ctfidf_model=topic_model.ctfidf_model,
    verbose=True
)

# Compute embeddings for the filtered documents
filtered_embeddings = embedding_model.encode(filtered_docs, show_progress_bar=True)

env_core_model.fit(documents=filtered_docs, embeddings=filtered_embeddings, y=filtered_new)

env_core_model.get_topic_info()

Old Topic 1 => 363 docs
Old Topic 5 => 193 docs
Old Topic 9 => 122 docs
Old Topic 14 => 78 docs
Docs kept: 756
Old topics kept: [1, 5, 9, 14]


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

2026-02-26 09:54:49,914 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-26 09:54:54,733 - BERTopic - Dimensionality - Completed ✓
2026-02-26 09:54:54,734 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-26 09:54:54,763 - BERTopic - Cluster - Completed ✓
2026-02-26 09:54:54,767 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-26 09:54:54,861 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,0,363,0_waste_chemicals_clothes_toxic,"[waste, chemicals, clothes, toxic, water, plas...",[well excerising helps body remove toxines bet...
1,1,193,1_fashion_fast_fast fashion_clothing,"[fashion, fast, fast fashion, clothing, sustai...",[fast fashion wreaking havoc environment produ...
2,2,122,2_sustainablefashion_clothing_sustainable_future,"[sustainablefashion, clothing, sustainable, fu...",[sustainable fashion plantbased dyes uses natu...
3,3,78,3_clothes_wear_wash_years,"[clothes, wear, wash, years, old, hand, use, c...",[chanelmindyabusiness yrs old many clothes yea...


In [ ]:
fig = env_core_model.visualize_barchart(
    topics=[0,1,2,3],
    n_words=10,
    title="Environment Core Topics (Relabeled 0–3)"
)

fig.write_html("topic_barchart_Environment_core_0_3.html")
print("✓ Saved: topic_barchart_Environment_core_0_3.html")
fig.show()


✓ Saved: topic_barchart_Environment_core_0_3.html
